In [1]:
import numpy as np


In [2]:

############################################
# 1. Define a super simple 2-site Hubbard model
############################################
# Hamiltonian:
# H = -t * sum_sigma ( c1† c2 + c2† c1 )
#     + U * (n1_up n1_dn + n2_up n2_dn)

t = 1.0   # hopping
U = 4.0   # on-site repulsion (correlation strength)

# We'll work in the full 2-electron singlet subspace with 2 sites.
# Basis states (ordered):
# |↑↓, 0>         (double occupancy site 1)
# |0, ↑↓>         (double occupancy site 2)
# |↑, ↓>_S        (one electron on each site, total Sz = 0 singlet combo)
#
# The last one is the spin singlet ( (|↑,↓>-|↓,↑>)/sqrt(2) ), which is the low-energy sector.
#
# In this basis, the Hamiltonian matrix is well-known for the half-filled 2-site Hubbard singlet block:

# We'll build it explicitly.
#
# State 0: |↑↓, 0>
# State 1: |0, ↑↓>
# State 2: |singlet (1e on each site)>
#
# Diagonal terms: U for double-occupied states, 0 for singly-occupied singlet.
#
# Off-diagonal hopping couples:
#  -t * sqrt(2) between |↑↓,0> <-> |singlet>
#  -t * sqrt(2) between |0,↑↓> <-> |singlet>
#
H = np.array([
    [ U,         0.0,        -np.sqrt(2)*t ],
    [ 0.0,       U,          -np.sqrt(2)*t ],
    [ -np.sqrt(2)*t, -np.sqrt(2)*t, 0.0    ],
], dtype=float)


In [3]:

############################################
# 2. Exact diagonalization (the real ground state)
############################################
evals, evecs = np.linalg.eigh(H)
E_exact = evals[0]
psi_exact = evecs[:,0]  # normalized ground state vector in our basis


In [4]:

############################################
# 3. Hartree–Fock energy for this system
############################################
# Mean-field HF for 2-site Hubbard basically assumes:
# replace n_up n_dn on each site with n_up⟨n_dn⟩ + n_dn⟨n_up⟩ - ⟨n_up⟩⟨n_dn⟩
#
# For half-filling and spin-symmetric solution, ⟨n_up⟩=⟨n_dn⟩=0.5 per site.
# Then per site U * (0.5*0.5) = U * 0.25 , and there are 2 sites -> U * 0.5 total.
#
# The HF effective one-body Hamiltonian is just hopping plus that constant.
#
E_HF_meanfield = U * 0.5 - 2*t*0.5  # rough analytic guess is not super clean
# Let's instead do self-consistent HF properly by iterating expectation values.

def hartree_fock_self_consistent(t, U, max_iter=1000, tol=1e-12):
    # Variables we solve for:
    # n1_up, n1_dn, n2_up, n2_dn.
    # We'll assume spin symmetry n1_up=n1_dn=n1/2, n2_up=n2_dn=n2/2
    # and total electrons = 2 => n1 + n2 = 2.
    #
    # With that symmetry we can just solve for n1 (electrons on site 1)
    # then n2 = 2 - n1.
    #
    # HF single-particle Hamiltonian for the spin-orbital "site basis":
    # h = [[ U * (n1/2),    -t        ],
    #      [    -t,     U * (n2/2)   ]]
    #
    # We'll fill two lowest spin-orbitals (since 2 electrons total, spin-degenerate),
    # which doubles the occupancy of each eigenvector.
    #
    # We'll iterate until n1 converges.

    n1 = 1.0  # start symmetric
    for _ in range(max_iter):
        n2 = 2.0 - n1

        h = np.array([
            [U * (n1/2.0),   -t],
            [   -t,          U * (n2/2.0)]
        ], dtype=float)

        eps, vecs = np.linalg.eigh(h)
        # lowest orbital is vecs[:,0]
        phi0 = vecs[:,0]  # length-2 vector giving amplitude on site1, site2

        # occupation: we put two electrons (up+down) into phi0
        # so density on site1 = 2 * |phi0[0]|^2
        n1_new = 2.0 * (np.abs(phi0[0])**2)

        if abs(n1_new - n1) < tol:
            n1 = n1_new
            break
        n1 = n1_new

    # final HF energy:
    # E = sum_occ eps_i  +  double-counting correction
    # we only occupied eps[0] with 2 electrons:
    E_onebody = 2.0 * eps[0]

    n2 = 2.0 - n1
    # interaction mean-field energy = U * [ n1_up n1_dn + n2_up n2_dn ]
    # = U * [ (n1/2)(n1/2) + (n2/2)(n2/2) ]
    E_int = U * ( (n1**2)/4.0 + (n2**2)/4.0 )

    E_HF = E_onebody - (U * ((n1/2.0)*(n1/2.0) + (n2/2.0)*(n2/2.0))) + E_int
    # Explanation:
    #   E_onebody already includes U*(n/2) terms in h, counted once per electron.
    #   We subtract the mean-field pieces we counted twice, then add the proper interaction energy.
    # For this symmetric 2-site 2-electron Hubbard, that reduces nicely, but let's keep it explicit.

    return E_HF, n1, n2, eps, phi0

E_HF, n1_final, n2_final, eps_final, phi0_final = hartree_fock_self_consistent(t, U)


In [5]:

############################################
# 4. Imaginary time evolution toward |psi_exact>
############################################
def imaginary_time_ground_state(H, tau_step=0.05, n_steps=200):
    # start from a random complex vector (not orthogonal to ground state)
    psi = np.random.randn(H.shape[0]) + 1j*np.random.randn(H.shape[0])
    psi = psi / np.linalg.norm(psi)

    energies = []
    for _ in range(n_steps):
        # propagate: psi <- (I - tau*H) psi
        psi = psi - tau_step * (H @ psi)

        # normalize
        psi = psi / np.linalg.norm(psi)

        # measure energy <psi|H|psi>
        E = np.real(np.vdot(psi, H @ psi))
        energies.append(E)

    return psi, np.array(energies)

psi_it, E_trace = imaginary_time_ground_state(H)

E_it_final = np.real(np.vdot(psi_it, H @ psi_it))


In [6]:

############################################
# 5. Print results
############################################
print("===== 2-site Hubbard demo =====")
print(f"Hubbard params: t={t}, U={U}")
print("")
print("Exact diagonalization:")
print(f"  E_exact = {E_exact:.6f} Ha")
print("")
print("Hartree-Fock (mean-field, restricted):")
print(f"  E_HF    = {E_HF:.6f} Ha")
print(f"  n1={n1_final:.4f}, n2={n2_final:.4f}  (should sum to 2)")
print("")
print("Imaginary-time projection:")
print(f"  E_it_final ≈ {E_it_final:.6f} Ha (after projection)")
print("")
print("Does imaginary-time state match exact ground state?")
overlap = abs(np.vdot(psi_exact, psi_it))
print(f"  |<psi_exact | psi_it>|^2 = {overlap**2:.6f}")
print("")
print("Does HF match exact?")
overlap_hf_guess = abs(np.vdot(psi_exact, np.array([1.0,0.0,0.0])))**2
print(f"  (toy HF-like double-occupancy overlap)^2 ≈ {overlap_hf_guess:.6f}")
print("")
print("Energy convergence of imaginary time (first 10 steps):")
print(E_trace[:10])
print("Energy convergence of imaginary time (last 10 steps):")
print(E_trace[-10:])

===== 2-site Hubbard demo =====
Hubbard params: t=1.0, U=4.0

Exact diagonalization:
  E_exact = -0.828427 Ha

Hartree-Fock (mean-field, restricted):
  E_HF    = 2.000000 Ha
  n1=1.0000, n2=1.0000  (should sum to 2)

Imaginary-time projection:
  E_it_final ≈ -0.828427 Ha (after projection)

Does imaginary-time state match exact ground state?
  |<psi_exact | psi_it>|^2 = 1.000000

Does HF match exact?
  (toy HF-like double-occupancy overlap)^2 ≈ 0.073223

Energy convergence of imaginary time (first 10 steps):
[ 3.05930166  2.42178648  1.69613232  0.98584814  0.38682224 -0.05810072
 -0.35850434 -0.54863679 -0.66419575 -0.73276888]
Energy convergence of imaginary time (last 10 steps):
[-0.82842712 -0.82842712 -0.82842712 -0.82842712 -0.82842712 -0.82842712
 -0.82842712 -0.82842712 -0.82842712 -0.82842712]
